# Fine-tune the encoder on a free GPU

This notebook trains DistilBERT on the NHTSA complaints, once for the random split and once for the time split.
It is written for Google Colab (the upload cell uses `google.colab`).

**Before you start**
1. Turn on a GPU: Colab, Runtime > Change runtime type > T4 GPU.
2. Have the file `data/processed/dataset.csv.gz` from your local copy of the repository ready. You upload it below.
3. Run the cells from top to bottom, and run each cell only once. Training took longer than 30 minutes per split in the second session, so plan for a long session.
4. At the end, download `finetune_results.zip` and unzip it in the folder of your local copy of the repository. The paths inside the zip already start with `results/`, so the files land in the right place.

The notebook uses the data that you upload and **checks that the splits are identical** to the ones in the repository.
If the check fails, stop: the results would not be comparable.

In [ ]:
REPO_URL = "https://github.com/Shashank-2902/complaint-classifier"   # change this if your repository has another address
!git clone {REPO_URL} complaint-classifier
%cd complaint-classifier
!pip install -q transformers pandas scikit-learn

### Why you upload the prepared data

NHTSA updates its complaint files every day. The data of this repository was downloaded on 2026-10-20.
If the notebook downloaded the files again, it would get slightly different data and therefore different splits, and the results could not be compared with the other systems.
So you upload the prepared file `data/processed/dataset.csv.gz` instead. The next two cells save it in the right place and compare a fingerprint of every split with the fingerprints
committed in `results/data_report.json`. If one differs, the check stops the notebook.

In [ ]:
import os, shutil
os.chdir("/content/complaint-classifier")
shutil.copy("results/data_report.json", "/content/committed_report.json")
from google.colab import files
uploaded = files.upload()
name = list(uploaded.keys())[0]
target = "/content/complaint-classifier/data/processed/dataset.csv.gz"
os.makedirs(os.path.dirname(target), exist_ok=True)
shutil.move(os.path.abspath(name), target)
print("saved:", name, "->", os.path.getsize(target), "bytes")

In [ ]:
import os, sys, json
os.chdir("/content/complaint-classifier")
sys.path.insert(0, "/content/complaint-classifier")
import pandas as pd
from src.prepare.run import fingerprint
frame = pd.read_csv("data/processed/dataset.csv.gz", dtype={"odi_no": str})
committed = json.load(open("/content/committed_report.json"))
same = all(fingerprint(frame.loc[frame[f"split_{s}"] == p, "odi_no"].tolist())
           == committed["splits"][s][p]["fingerprint"]
           for s in ("random", "time") for p in ("train", "val", "test"))
print("splits identical to the repository:", same, "| rows:", len(frame))
assert same, "The uploaded dataset does not match the repository. Do not continue."

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# Quick check that everything works (a few hundred complaints, never touches the test set).
!python -m src.finetune.train --scheme random --smoke

### The real training runs

Run each of the next two cells only once. Each one opens the test set once, at the end. Running a cell a second time would be a second use of the test set.

In [ ]:
# Random split. Opens the random test set once, at the end.
!python -m src.finetune.train --scheme random

In [ ]:
# Time split. Opens the time test set once, at the end.
!python -m src.finetune.train --scheme time

In [ ]:
# Pack the results to download. This session's access log shows when each test set was opened here. It cannot see other sessions.
!grep finetuned_encoder results/test_access_log.jsonl > results/test_access_log_finetuned.jsonl
!zip -r finetune_results.zip results/predictions/finetuned_encoder__*.csv results/predictions/finetuned_encoder__*.meta.json results/test_access_log_finetuned.jsonl
from google.colab import files
files.download("finetune_results.zip")